# Advanced Grape Leaf Disease Analysis: Segmentation-Driven Classification

This notebook implements a multi-stage pipeline for grape leaf disease diagnosis:
1. **Segmentation Options**: 
   - *Traditional*: Robust color-space analysis (HSV/Lab).
   - *Deep Learning*: Architectures for U-Net, DeepLabV3+, and FCN-8s.
2. **Segmented Classification**: Disease classification performed on the isolated leaf area to improve accuracy.
3. **Ensemble Learning**: Combining DenseNet and EfficientNet for classification.
4. **Severity Analysis**: Calculating the percentage of infection on the isolated leaf surface.

In [ ]:
# !pip install torch torchvision torchaudio scikit-learn matplotlib opencv-python pillow

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix
from PIL import Image
import copy

# Optional: Mount Google Drive if using Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except:
    print("Not running in Google Colab or Drive mounting failed.")

## 0. Configuration and Data Loading

In [ ]:
DATASET_PATH = 'path_to_your_dataset' # E.g. '/content/drive/MyDrive/GrapeDataset'
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

class SegmentedGrapeDataset(Dataset):
    """Dataset that applies leaf segmentation during training."""
    def __init__(self, data_dir, transform=None):
        self.dataset = datasets.ImageFolder(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        path, label = self.dataset.samples[idx]
        
        # Perform segmentation (leaf isolation)
        image = cv2.imread(path)
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        lower_green = np.array([25, 40, 40])
        upper_green = np.array([90, 255, 255])
        mask = cv2.inRange(hsv, lower_green, upper_green)
        leaf_only = cv2.bitwise_and(image, image, mask=mask)
        leaf_only_rgb = cv2.cvtColor(leaf_only, cv2.COLOR_BGR2RGB)
        
        pil_img = Image.fromarray(leaf_only_rgb)
        if self.transform:
            pil_img = self.transform(pil_img)
        
        return pil_img, label

def get_dataloaders(data_dir):
    transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    full_dataset = SegmentedGrapeDataset(data_dir, transform=transform)
    labels = [s[1] for s in full_dataset.dataset.samples]
    
    # Stratified split: 70% Train, 15% Val, 15% Test
    train_idx, temp_idx = train_test_split(
        np.arange(len(full_dataset)), test_size=0.3, stratify=labels, random_state=42
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, stratify=[labels[i] for i in temp_idx], random_state=42
    )
    
    loaders = {
        'train': DataLoader(Subset(full_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True),
        'val': DataLoader(Subset(full_dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False),
        'test': DataLoader(Subset(full_dataset, test_idx), batch_size=BATCH_SIZE, shuffle=False)
    }
    return loaders, full_dataset.dataset.classes

# loaders, class_names = get_dataloaders(DATASET_PATH)

## 1. Dual Segmentation Module

### 1.1 Traditional Segmentation (HSV/Lab)

## 5. Execution Loop (Training and Comparison)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize Loaders
# loaders, class_names = get_dataloaders(DATASET_PATH)
# num_classes = len(class_names)

# 2. Baseline: DenseNet
# densenet = get_densenet_model(num_classes)
# densenet = train_model(densenet, loaders, nn.CrossEntropyLoss(), optim.AdamW(densenet.parameters(), lr=0.001), device=device)

# 3. Baseline: EfficientNet
# efficientnet = get_efficientnet_model(num_classes)
# efficientnet = train_model(efficientnet, loaders, nn.CrossEntropyLoss(), optim.AdamW(efficientnet.parameters(), lr=0.001), device=device)

# 4. Proposed Ensemble (Soft Voting)
# ensemble = ClassificationEnsemble(densenet, efficientnet, num_classes)

# 5. Performance Comparison
# evaluate_and_plot_roc({'DenseNet': densenet, 'EfficientNet': efficientnet, 'Ensemble': ensemble}, loaders['test'], class_names, device=device)

## 6. Full Integrated Pipeline Inference

The pipeline follows: **Original -> Segmentation (Leaf Isolation) -> Classification of Segmented Image -> Severity Calculation.**

In [ ]:
def traditional_segment_leaf(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Green range for leaf
    lower_green = np.array([25, 40, 40])
    upper_green = np.array([90, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    
    # Refine
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    leaf_only = cv2.bitwise_and(image_rgb, image_rgb, mask=mask)
    return image_rgb, mask, leaf_only

def traditional_segment_disease(leaf_rgb, leaf_mask):
    lab = cv2.cvtColor(leaf_rgb, cv2.COLOR_RGB2Lab)
    _, a, _ = cv2.split(lab)
    a_blurred = cv2.GaussianBlur(a, (5, 5), 0)
    _, disease_mask = cv2.threshold(a_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    disease_mask = cv2.bitwise_and(disease_mask, disease_mask, mask=leaf_mask)
    return disease_mask

### 1.2 Deep Learning Segmentation (Architectures)
Note: These require training on pixel-level masks.

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(SimpleUNet, self).__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True)
            )
        self.enc1 = conv_block(in_channels, 64)
        self.enc2 = conv_block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = conv_block(128, 64)
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        u1 = self.up1(e2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return torch.sigmoid(self.final(d1))

def get_deeplabv3_model(num_classes=1):
    model = models.segmentation.deeplabv3_resnet50(pretrained=True)
    model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
    return model

def get_fcn_model(num_classes=1):
    model = models.segmentation.fcn_resnet50(pretrained=True)
    model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
    return model

## 2. Classification Module (Ensemble)

We use **DenseNet121** and **EfficientNet_B0** for the classification ensemble.

In [ ]:
def get_densenet_model(num_classes):
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model

def get_efficientnet_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

class ClassificationEnsemble(nn.Module):
    def __init__(self, modelA, modelB, num_classes):
        super(ClassificationEnsemble, self).__init__()
        self.modelA = modelA
        self.modelB = modelB
        # Use identity for classifier to get features or keep for soft voting
        
    def forward(self, x):
        out1 = self.modelA(x)
        out2 = self.modelB(x)
        return (out1 + out2) / 2 # Soft voting (average of logits)

## 3. Training and Evaluation Logic

In [ ]:
def train_model(model, loaders, criterion, optimizer, num_epochs=10, device='cuda'):
    model = model.to(device)
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        for phase in ['train', 'val']:
            if phase == 'train': model.train()
            else: model.eval()

            running_loss, running_corrects = 0.0, 0
            for inputs, labels in loaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            epoch_acc = running_corrects.double() / len(loaders[phase].dataset)
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print(f'Epoch {epoch} Val Acc: {best_acc:.4f}')
    
    model.load_state_dict(best_model_wts)
    return model

def evaluate_and_plot_roc(models_dict, test_loader, class_names, device='cuda'):
    plt.figure(figsize=(10, 8))
    for name, model in models_dict.items():
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                outputs = F.softmax(model(inputs), dim=1)
                all_labels.extend(labels.numpy())
                all_probs.extend(outputs.cpu().numpy())
        
        all_labels = np.array(all_labels)
        all_probs = np.array(all_probs)
        
        # Micro-average ROC
        fpr, tpr, _ = roc_curve(np.eye(len(class_names))[all_labels].ravel(), all_probs.ravel())
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc(fpr, tpr):.2f})')
    
    plt.plot([0,1],[0,1], 'k--')
    plt.legend()
    plt.title("ROC Curve Comparison")
    plt.show()

## 4. Full Integrated Pipeline

The pipeline follows: **Original -> Segmentation (Leaf Isolation) -> Classification of Segmented Image -> Severity Calculation.**

In [ ]:
def pipeline_inference(image_path, classification_model, class_names, device='cuda'):
    # 1. Segmentation (Leaf Isolation)
    orig_rgb, leaf_mask, leaf_only = traditional_segment_leaf(image_path)
    
    # 2. Classification of Segmented Result
    leaf_pil = Image.fromarray(leaf_only)
    preprocess = transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    input_tensor = preprocess(leaf_pil).unsqueeze(0).to(device)
    
    classification_model.eval()
    with torch.no_grad():
        logits = classification_model(input_tensor)
        _, pred = torch.max(logits, 1)
        label = class_names[pred[0]]
    
    # 3. Disease Segmentation & Severity
    disease_mask = traditional_segment_disease(leaf_only, leaf_mask)
    severity = (np.sum(disease_mask > 0) / np.sum(leaf_mask > 0)) * 100 if np.sum(leaf_mask > 0) > 0 else 0
    
    # 4. Visualization
    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    ax[0].imshow(orig_rgb); ax[0].set_title("1. Original Image")
    ax[1].imshow(leaf_only); ax[1].set_title("2. Isolated Leaf (Segmented)")
    ax[2].imshow(disease_mask, cmap='hot'); ax[2].set_title("3. Disease Lesions")
    ax[3].imshow(leaf_only)
    overlay = leaf_only.copy()
    overlay[disease_mask > 0] = [255, 0, 0]
    ax[3].imshow(overlay)
    ax[3].set_title(f"4. Final: {label}\nSeverity: {severity:.2f}%")
    plt.tight_layout()
    plt.show()